In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rakibullah/mahdy-research-academy-all-datasets/LOS dataset/LOS dataset/1000 instances  Chatgpt Los/1000 instances train Chatgpt Los.csv
/kaggle/input/datasets/rakibullah/mahdy-research-academy-all-datasets/LOS dataset/LOS dataset/1000 instances  Chatgpt Los/1000 instances test Chatgpt Los.csv
/kaggle/input/datasets/rakibullah/mahdy-research-academy-all-datasets/LOS dataset/LOS dataset/1000 instances  Chatgpt Los/1000 instances val Chatgpt Los.csv
/kaggle/input/datasets/rakibullah/mahdy-research-academy-all-datasets/LOS dataset/LOS dataset/BanglaNMT_Los All/BanglaNMT_Los All/BanglaNMT_Los/BanglaNMT_Los/LOS_adm_test_BanglaNmt.csv
/kaggle/input/datasets/rakibullah/mahdy-research-academy-all-datasets/LOS dataset/LOS dataset/BanglaNMT_Los All/BanglaNMT_Los All/BanglaNMT_Los/BanglaNMT_Los/LOS_adm_val_BanglaNmt.csv
/kaggle/input/datasets/rakibullah/mahdy-research-academy-all-datasets/LOS dataset/LOS dataset/BanglaNMT_Los All/BanglaNMT_Los All/BanglaNMT_Los/BanglaNMT_Los

In [7]:
!pip install sacremoses
import time
import os
import torch
import pandas as pd

# ── Load data ───────────────────────────────────────────────────────────────
val = pd.read_excel('/kaggle/input/datasets/rakibullah/diiiiiiiiiiiiii/Prompt Val 100.xlsx')
val['text'] = val['text'].str.lower()

# ── Model size on disk ──────────────────────────────────────────────────────
from huggingface_hub import snapshot_download
model_path = snapshot_download("shhossain/opus-mt-en-to-bn")
model_size_bytes = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, filenames in os.walk(model_path)
    for f in filenames
)
model_size_mb = model_size_bytes / (1024 ** 2)

# ── Number of parameters ────────────────────────────────────────────────────
from transformers import MarianMTModel, MarianTokenizer

model = MarianMTModel.from_pretrained("shhossain/opus-mt-en-to-bn")
num_params = sum(p.numel() for p in model.parameters())
num_params_m = num_params / 1e6

# ── Inference time (average over 5 samples from val set) ───────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = MarianTokenizer.from_pretrained("shhossain/opus-mt-en-to-bn")
model = model.to(device)
model.eval()

sample_texts = val['text'].iloc[:5].tolist()
times = []
for text in sample_texts:
    inputs = tokenizer(text[:800], return_tensors="pt", padding=True, truncation=True).to(device)
    start = time.time()
    with torch.no_grad():
        translated = model.generate(**inputs)
    times.append(time.time() - start)

avg_inference_sec = sum(times) / len(times)

# ── Print summary ───────────────────────────────────────────────────────────
print(f"Model Size      : {model_size_mb:.1f} MB")
print(f"Parameters      : {num_params_m:.1f}M")
print(f"Avg Inference   : {avg_inference_sec:.3f} sec / 800-char chunk")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model Size      : 582.2 MB
Parameters      : 76.3M
Avg Inference   : 0.939 sec / 800-char chunk


train['text'][980]

In [3]:
!pip install sacremoses

import time
import os
import torch
import pandas as pd
from huggingface_hub import snapshot_download
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm

# ── Data ─────────────────────────────────────────────────────────────────────
train = pd.read_csv('/kaggle/input/mydata/LOS_WEEKS_adm_train.csv')
val   = pd.read_csv('/kaggle/input/mydata/LOS_WEEKS_adm_val.csv')
test  = pd.read_csv('/kaggle/input/mydata/LOS_WEEKS_adm_test.csv')
for df in [train, val, test]:
    df['text'] = df['text'].str.lower()

# ── Model size on disk ────────────────────────────────────────────────────────
model_path = snapshot_download("csebuetnlp/banglat5_nmt_en_bn")
model_size_bytes = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, filenames in os.walk(model_path)
    for f in filenames
)
model_size_mb = model_size_bytes / (1024 ** 2)

# ── Load model & tokenizer ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglat5_nmt_en_bn")
model     = AutoModelForSeq2SeqLM.from_pretrained("csebuetnlp/banglat5_nmt_en_bn").to(device)
model.eval()

num_params_m = sum(p.numel() for p in model.parameters()) / 1e6

# ── Avg inference time (5 samples) ───────────────────────────────────────────
sample_texts = val['text'].iloc[:5].tolist()
times = []
for text in sample_texts:
    inputs = tokenizer(text[:800], return_tensors="pt", padding=True, truncation=True).to(device)
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_length=1000)
    times.append(time.time() - start)
avg_inference_sec = sum(times) / len(times)

print(f"Model Size      : {model_size_mb:.1f} MB")
print(f"Parameters      : {num_params_m:.1f}M")
print(f"Avg Inference   : {avg_inference_sec:.3f} sec / 800-char chunk")

# ── Translation ───────────────────────────────────────────────────────────────
SENTINEL = 'কিন্তু, তা সত্ত্বেও তিনি যিহোবার প্রতি অনুগত ছিলেন।'
CHUNK    = 800

final = []
print('Range 18000:22000')

for text in tqdm(train['text'][18000:22000]):
    chunks, start = [], 0
    while True:
        chunk_text = text[start: start + CHUNK]
        if not chunk_text:
            break
        inputs = tokenizer(chunk_text, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_length=1000)
        result = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        if result == SENTINEL:
            break
        chunks.append(result)
        start += CHUNK
    final.append(" ".join(chunks))

print(f"Translated: {len(final)}")

# ── Save ──────────────────────────────────────────────────────────────────────
trans = pd.DataFrame({'machine': final})
trans.to_csv('BanglaNMT_Train_18000_22000_LOS.csv', index=False)

Using cuda


config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

KeyError: "Invalid translation task translation, use 'translation_XX_to_YY' format"

In [4]:
import os, time, torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = {
    "opus-mt-en-bn": {
        "hf_name": "shhossain/opus-mt-en-to-bn",
        "model_cls": MarianMTModel,
        "tok_cls":   MarianTokenizer,
    },
    "banglat5-nmt": {
        "hf_name": "csebuetnlp/banglat5_nmt_en_bn",
        "model_cls": AutoModelForSeq2SeqLM,
        "tok_cls":   AutoTokenizer,
    },
}

sample_text = "the patient was admitted with chest pain and shortness of breath."

results = {}
for name, cfg in models.items():
    # Model size on disk
    path = snapshot_download(cfg["hf_name"])
    size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(path) for f in files
    ) / (1024 ** 2)

    # Params
    model = cfg["model_cls"].from_pretrained(cfg["hf_name"]).to(device)
    model.eval()
    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    # Inference time (avg over 5 runs)
    tok = cfg["tok_cls"].from_pretrained(cfg["hf_name"])
    inputs = tok(sample_text, return_tensors="pt", truncation=True).to(device)
    times = []
    for _ in range(5):
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_length=512)
        times.append(time.time() - start)
    avg_time = sum(times) / len(times)

    results[name] = {"size_mb": size_mb, "params_m": params_m, "avg_inference_sec": avg_time}
    del model

for name, r in results.items():
    print(f"\n{name}")
    print(f"  Model Size    : {r['size_mb']:.1f} MB")
    print(f"  Parameters    : {r['params_m']:.1f}M")
    print(f"  Avg Inference : {r['avg_inference_sec']:.3f} sec")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]


opus-mt-en-bn
  Model Size    : 582.2 MB
  Parameters    : 76.3M
  Avg Inference : 0.476 sec

banglat5-nmt
  Model Size    : 945.6 MB
  Parameters    : 296.9M
  Avg Inference : 0.443 sec


In [8]:
import os, time, torch
import pandas as pd
from huggingface_hub import snapshot_download
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

val = pd.read_excel('/kaggle/input/datasets/rakibullah/diiiiiiiiiiiiii/Prompt Val 100.xlsx')
val['text'] = val['text'].str.lower()
sample_texts = val['text'].iloc[:10].tolist()

models = {
    "opus-mt-en-bn": {
        "hf_name":   "shhossain/opus-mt-en-to-bn",
        "model_cls": MarianMTModel,
        "tok_cls":   MarianTokenizer,
    },
    "banglat5-nmt": {
        "hf_name":   "csebuetnlp/banglat5_nmt_en_bn",
        "model_cls": AutoModelForSeq2SeqLM,
        "tok_cls":   AutoTokenizer,
    },
}

results = {}
for name, cfg in models.items():
    # Model size on disk
    path = snapshot_download(cfg["hf_name"])
    size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(path) for f in files
    ) / (1024 ** 2)

    # Load
    tok   = cfg["tok_cls"].from_pretrained(cfg["hf_name"])
    model = cfg["model_cls"].from_pretrained(cfg["hf_name"]).to(device)
    model.eval()
    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    # Avg inference over 5 samples, 800-char chunk each
    times = []
    for text in sample_texts:
        inputs = tok(text[:800], return_tensors="pt", padding=True, truncation=True).to(device)
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_length=3000)
        times.append(time.time() - start)
    avg_time = sum(times) / len(times)

    results[name] = {"size_mb": size_mb, "params_m": params_m, "avg_inference_sec": avg_time}
    del model

for name, r in results.items():
    print(f"\n{name}")
    print(f"  Model Size    : {r['size_mb']:.1f} MB")
    print(f"  Parameters    : {r['params_m']:.1f}M")
    print(f"  Avg Inference : {r['avg_inference_sec']:.3f} sec / 800-char chunk")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



opus-mt-en-bn
  Model Size    : 582.2 MB
  Parameters    : 76.3M
  Avg Inference : 1.030 sec / 800-char chunk

banglat5-nmt
  Model Size    : 945.6 MB
  Parameters    : 296.9M
  Avg Inference : 2.718 sec / 800-char chunk


In [10]:
import os, time, torch
import pandas as pd
from huggingface_hub import snapshot_download
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

val = pd.read_excel('/kaggle/input/datasets/rakibullah/diiiiiiiiiiiiii/Prompt Val 100.xlsx')
val['text'] = val['text'].str.lower()
sample_texts = val['text'].iloc[:5].tolist()

models = {
    "opus-mt-en-bn": {
        "hf_name":   "shhossain/opus-mt-en-to-bn",
        "model_cls": MarianMTModel,
        "tok_cls":   MarianTokenizer,
    },
    "banglat5-nmt": {
        "hf_name":   "csebuetnlp/banglat5_nmt_en_bn",
        "model_cls": AutoModelForSeq2SeqLM,
        "tok_cls":   AutoTokenizer,
    },
}

results = {}
for name, cfg in models.items():
    # Model size on disk
    path = snapshot_download(cfg["hf_name"])
    size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(path) for f in files
    ) / (1024 ** 2)

    # Load
    tok   = cfg["tok_cls"].from_pretrained(cfg["hf_name"])
    model = cfg["model_cls"].from_pretrained(cfg["hf_name"]).to(device)
    model.eval()
    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    # Warmup pass to avoid first-run CUDA kernel overhead
    with torch.no_grad():
        warmup_input = tok("warmup", return_tensors="pt", truncation=True).to(device)
        model.generate(**warmup_input, max_new_tokens=5)

    # Avg inference over 5 samples, 800-char chunk each
    times = []
    for text in sample_texts:
        inputs = tok(text[:800], return_tensors="pt", padding=True, truncation=True).to(device)
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_length=512)
        times.append(time.time() - start)

    avg_time = sum(times) / len(times)
    results[name] = {"size_mb": size_mb, "params_m": params_m, "avg_inference_sec": avg_time}
    del model

for name, r in results.items():
    print(f"\n{name}")
    print(f"  Model Size    : {r['size_mb']:.1f} MB")
    print(f"  Parameters    : {r['params_m']:.1f}M")
    print(f"  Avg Inference : {r['avg_inference_sec']:.3f} sec / 800-char chunk")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



opus-mt-en-bn
  Model Size    : 582.2 MB
  Parameters    : 76.3M
  Avg Inference : 1.019 sec / 800-char chunk

banglat5-nmt
  Model Size    : 945.6 MB
  Parameters    : 296.9M
  Avg Inference : 2.754 sec / 800-char chunk


In [13]:
import os, time, torch

import pandas as pd

from huggingface_hub import snapshot_download

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

val = pd.read_excel('/kaggle/input/datasets/rakibullah/diiiiiiiiiiiiii/Prompt Val 100.xlsx')

val['text'] = val['text'].str.lower()

sample_texts = val['text'].iloc[:5].tolist()

models = {

    "opus-mt-en-bn": {

        "hf_name":   "shhossain/opus-mt-en-to-bn",

        "model_cls": MarianMTModel,

        "tok_cls":   MarianTokenizer,

        "chunk_size": 300,

    },

    "banglat5-nmt": {

        "hf_name":   "csebuetnlp/banglat5_nmt_en_bn",

        "model_cls": AutoModelForSeq2SeqLM,

        "tok_cls":   AutoTokenizer,

        "chunk_size": 800,

    },

}

results = {}

for name, cfg in models.items():

    # Model size on disk

    path = snapshot_download(cfg["hf_name"])

    size_mb = sum(

        os.path.getsize(os.path.join(dp, f))

        for dp, _, files in os.walk(path) for f in files

    ) / (1024 ** 2)

    # Load

    tok   = cfg["tok_cls"].from_pretrained(cfg["hf_name"])

    model = cfg["model_cls"].from_pretrained(cfg["hf_name"]).to(device)

    model.eval()

    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    

    # Warmup pass to avoid first-run CUDA kernel overhead
    warmup_iterations = 3
    for _ in range(warmup_iterations):
        with torch.no_grad():
            warmup_input = tok("warmup text sample", return_tensors="pt", truncation=True).to(device)
            model.generate(**warmup_input, max_new_tokens=20)

    # Avg inference over 5 samples with model-specific chunk size

    chunk_size = cfg.get("chunk_size", 800)

    times = []

    for text in sample_texts:

        inputs = tok(text[:chunk_size], return_tensors="pt", padding=True, truncation=True).to(device)

        start = time.time()

        with torch.no_grad():

            model.generate(**inputs, max_length=3000)

        times.append(time.time() - start)

    avg_time = sum(times) / len(times)

    results[name] = {"size_mb": size_mb, "params_m": params_m, "avg_inference_sec": avg_time, "chunk_size": chunk_size}

    del model

for name, r in results.items():

    print(f"\n{name}")

    print(f"  Chunk Size     : {r['chunk_size']} chars")

    print(f"  Model Size    : {r['size_mb']:.1f} MB")

    print(f"  Parameters    : {r['params_m']:.1f}M")

    print(f"  Avg Inference : {r['avg_inference_sec']:.3f} sec / {r['chunk_size']}-char chunk")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



opus-mt-en-bn
  Chunk Size     : 300 chars
  Model Size    : 582.2 MB
  Parameters    : 76.3M
  Avg Inference : 0.844 sec / 300-char chunk

banglat5-nmt
  Chunk Size     : 800 chars
  Model Size    : 945.6 MB
  Parameters    : 296.9M
  Avg Inference : 2.765 sec / 800-char chunk


In [4]:
import os, time, torch
import pandas as pd
from tqdm import tqdm

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load data
val = pd.read_excel('/kaggle/input/datasets/rakibullah/diiiiiiiiiiiiii/Prompt Val 100.xlsx')
val['text'] = val['text'].str.lower()

sample_texts = val['text'].iloc[:5].tolist()

# Google Translate function with chunk handling
def translate_google(text, chunk_size=5000):
    """
    Translate text using Google Translate with chunk processing.
    Uses bettergoogletrans library for reliability.
    """
    try:
        from googletrans import Translator
    except ImportError:
        print("Installing googletrans...")
        os.system("pip install googletrans")
        from googletrans import Translator
    
    translator = Translator()
    initial = []
    start = 0
    finish = chunk_size
    
    for i in range(9999999999):
        try:
            translation = translator.translate(text, src='en', dest='bn')
            return translation.text
        except TypeError:
            try:
                translation = translator.translate(text[start:finish], src='en', dest='bn')
                initial.append(translation.text)
                start = finish
                finish = finish + chunk_size
            except TypeError:
                return " ".join(initial)

'''# Translation models configuration
from huggingface_hub import snapshot_download
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

models = {
    "opus-mt-en-bn": {
        "hf_name": "shhossain/opus-mt-en-to-bn",
        "model_cls": MarianMTModel,
        "tok_cls": MarianTokenizer,
        "chunk_size": 300,
    },
    "banglat5-nmt": {
        "hf_name": "csebuetnlp/banglat5_nmt_en_bn",
        "model_cls": AutoModelForSeq2SeqLM,
        "tok_cls": AutoTokenizer,
        "chunk_size": 800,
    },
}

results = {}

# Benchmark Hugging Face models
for name, cfg in models.items():
    print(f"\nBenchmarking {name}...")
    
    # Model size on disk
    path = snapshot_download(cfg["hf_name"])
    size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(path) for f in files
    ) / (1024 ** 2)
    
    # Load model
    tok = cfg["tok_cls"].from_pretrained(cfg["hf_name"])
    model = cfg["model_cls"].from_pretrained(cfg["hf_name"]).to(device)
    model.eval()
    
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    
    # Warmup pass
    with torch.no_grad():
        warmup_input = tok("warmup", return_tensors="pt", truncation=True).to(device)
        model.generate(**warmup_input, max_new_tokens=5)
    
    # Benchmark inference
    chunk_size = cfg.get("chunk_size", 800)
    times = []
    for text in sample_texts:
        inputs = tok(text[:chunk_size], return_tensors="pt", padding=True, truncation=True).to(device)
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_length=512)
        times.append(time.time() - start)
    
    avg_time = sum(times) / len(times)
    results[name] = {"size_mb": size_mb, "params_m": params_m, "avg_inference_sec": avg_time, "chunk_size": chunk_size}
    
    del model
    torch.cuda.empty_cache()'''

# Benchmark Google Translate
print("\nBenchmarking Google Translate...")
chunk_size = 300  # Same chunk size as opus for fair comparison
times = []
for text in tqdm(sample_texts, desc="Google Translate"):
    start = time.time()
    translate_google(text[:chunk_size])
    times.append(time.time() - start)

avg_time = sum(times) / len(times)
results["google-translate"] = {"size_mb": 0, "params_m": 0, "avg_inference_sec": avg_time, "chunk_size": chunk_size}

# Print results
print("\n" + "="*60)
print("TRANSLATION BENCHMARK RESULTS")
print("="*60)
for name, r in results.items():
    print(f"\n{name}")
    print(f"  Chunk Size     : {r['chunk_size']} chars")
    print(f"  Model Size     : {r['size_mb']:.1f} MB" if r['size_mb'] > 0 else "  Model Size     : N/A")
    print(f"  Parameters     : {r['params_m']:.1f}M" if r['params_m'] > 0 else "  Parameters     : N/A")
    print(f"  Avg Inference  : {r['avg_inference_sec']:.3f} sec / {r['chunk_size']}-char chunk")
print("="*60)


Benchmarking Google Translate...



Google Translate:   0%|          | 0/5 [00:00<?, ?it/s]


AttributeError: module 'httpcore' has no attribute 'SyncHTTPTransport'

# Latest

In [8]:
import os, time, torch
import pandas as pd
from tqdm import tqdm

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load data
val = pd.read_excel('/kaggle/input/datasets/rakibullah/diiiiiiiiiiiiii/Prompt Val 100.xlsx')
val['text'] = val['text'].str.lower()

sample_texts = val['text'].iloc[:5].tolist()

# Google Translate function with chunk handling
def translate_google(text, chunk_size=5000):
    """
    Translate text using Google Translate with chunk processing.
    Uses deep_translator library for reliability (fixes googletrans compatibility issues).
    """
    try:
        from deep_translator import GoogleTranslator
    except ImportError:
        print("Installing deep_translator...")
        os.system("pip install deep-translator")
        from deep_translator import GoogleTranslator
    
    initial = []
    start = 0
    finish = chunk_size
    
    try:
        # Try full translation first
        translator = GoogleTranslator(source='en', target='bn')
        translation = translator.translate(text)
        return translation
    except Exception:
        # Fall back to chunked translation
        try:
            translator = GoogleTranslator(source='en', target='bn')
            while start < len(text):
                chunk = text[start:finish]
                translation = translator.translate(chunk)
                initial.append(translation)
                start = finish
                finish = finish + chunk_size
            return " ".join(initial)
        except Exception as e:
            print(f"Translation error: {e}")
            return " ".join(initial)

# Translation models configuration
from huggingface_hub import snapshot_download
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

models = {
    "opus-mt-en-bn": {
        "hf_name": "shhossain/opus-mt-en-to-bn",
        "model_cls": MarianMTModel,
        "tok_cls": MarianTokenizer,
        "chunk_size": 300,
    },
    "banglat5-nmt": {
        "hf_name": "csebuetnlp/banglat5_nmt_en_bn",
        "model_cls": AutoModelForSeq2SeqLM,
        "tok_cls": AutoTokenizer,
        "chunk_size": 800,
    },
}

results = {}

# Benchmark Hugging Face models
for name, cfg in models.items():
    print(f"\nBenchmarking {name}...")
    
    # Model size on disk
    path = snapshot_download(cfg["hf_name"])
    size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(path) for f in files
    ) / (1024 ** 2)
    
    # Load model
    tok = cfg["tok_cls"].from_pretrained(cfg["hf_name"])
    model = cfg["model_cls"].from_pretrained(cfg["hf_name"]).to(device)
    model.eval()
    
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    
    # Warmup pass
    with torch.no_grad():
        warmup_input = tok("warmup", return_tensors="pt", truncation=True).to(device)
        model.generate(**warmup_input, max_new_tokens=5)
    
    # Benchmark inference
    chunk_size = cfg.get("chunk_size", 800)
    times = []
    for text in sample_texts:
        inputs = tok(text[:chunk_size], return_tensors="pt", padding=True, truncation=True).to(device)
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_length=512)
        times.append(time.time() - start)
    
    avg_time = sum(times) / len(times)
    results[name] = {"size_mb": size_mb, "params_m": params_m, "avg_inference_sec": avg_time, "chunk_size": chunk_size}
    
    del model
    torch.cuda.empty_cache()

# Benchmark Google Translate
print("\nBenchmarking Google Translate...")
chunk_size = 300  # Same chunk size as opus for fair comparison
times = []
for text in tqdm(sample_texts, desc="Google Translate"):
    start = time.time()
    translate_google(text[:chunk_size])
    times.append(time.time() - start)

avg_time = sum(times) / len(times)
results["google-translate"] = {"size_mb": 0, "params_m": 0, "avg_inference_sec": avg_time, "chunk_size": chunk_size}

# Print results
print("\n" + "="*60)
print("TRANSLATION BENCHMARK RESULTS")
print("="*60)
for name, r in results.items():
    print(f"\n{name}")
    print(f"  Chunk Size     : {r['chunk_size']} chars")
    print(f"  Model Size     : {r['size_mb']:.1f} MB" if r['size_mb'] > 0 else "  Model Size     : N/A")
    print(f"  Parameters     : {r['params_m']:.1f}M" if r['params_m'] > 0 else "  Parameters     : N/A")
    print(f"  Avg Inference  : {r['avg_inference_sec']:.3f} sec / {r['chunk_size']}-char chunk")
print("="*60)


Benchmarking opus-mt-en-bn...


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


Benchmarking banglat5-nmt...


...

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Benchmarking Google Translate...



Google Translate: 100%|██████████| 5/5 [00:00<00:00,  9.52it/s]


TRANSLATION BENCHMARK RESULTS

opus-mt-en-bn
  Chunk Size     : 300 chars
  Model Size     : 582.2 MB
  Parameters     : 76.3M
  Avg Inference  : 0.729 sec / 300-char chunk

banglat5-nmt
  Chunk Size     : 800 chars
  Model Size     : 945.6 MB
  Parameters     : 296.9M
  Avg Inference  : 2.485 sec / 800-char chunk

google-translate
  Chunk Size     : 300 chars
  Model Size     : N/A
  Parameters     : N/A
  Avg Inference  : 0.104 sec / 300-char chunk
